In [1]:
import os, sys
sys.path.append(os.path.abspath("F:/Neutrino_SI/Pycode"))  # 让 Python 能找到该目录的模块

In [16]:
from Pycode import definition as df
import numpy as np
import matplotlib.pyplot as plt
from Pycode import SignalRate_1d as SignalRate
import Data
from scipy import integrate
from Pycode import Errors
from Simulation_Spectrum import time_limit

t_max = time_limit('max')
E_max = df.E_max_2d

E_min_K = df.E_min_K
E_min_I = df.E_min_I
E_min_B = df.E_min_B

ranges_K = [(0, t_max), (E_min_K, E_max), (-1, 1)]     

ranges_I = [(0, t_max), (E_min_I, E_max), (-1, 1)]

ranges_B = [(0, t_max), (E_min_B, E_max), (-1, 1)]

percision = 64

def log_likelihood_K(theta, t_K, E_K, c_K, dE_K, B_K):

    # R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu = theta

    R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu = theta

    fun1_K = lambda x: SignalRate.SR_K(t_K, x, c_K, *[R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu]) * Errors.Error_E(x, E_K, dE_K)        # optional

    part1_K = df.gl3_integrate(SignalRate.SR_K, ranges_K, percision, args=[R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu])

    part2_K = df.gl_integrate(fun1_K, E_min_K, E_max)

    part3_K = B_K/2

    ret_K = -part1_K + np.sum(np.log(part3_K +part2_K))

    return ret_K


# IMB
def log_likelihood_I(theta, t_I, E_I, c_I, dE_I, B_I):

    R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu = theta
    
    fun1_I = lambda x: SignalRate.SR_I(t_I, x, c_I, *[R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu])*Errors.Error_E(x, E_I, dE_I)

    fun2_I = lambda x, c, t: SignalRate.SR_I(t, x, c, *[R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu])*Errors.Error_E(x, E_I, dE_I)

    part1_I = df.gl3_integrate(SignalRate.SR_I, ranges_I, percision, args=[R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu])

    part2_I = np.clip(df.gl_integrate(fun1_I, E_min_I , E_max), 1e-100, None)

    part3_I = B_I/2

    part4_I= df.gl2_integrate_vec(fun2_I, ((E_min_I, E_max), (-1,1)), t_I, percision)

    ret_I = -0.9055*part1_I  + np.sum(0.035*part4_I) + np.sum(np.log(part3_I + part2_I))

    return ret_I

# Baksan
def log_likelihood_B(theta, t_B, E_B, c_B, dE_B, B_B):

    R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu= theta

    fun1_B = lambda x: SignalRate.SR_B(t_B, x, c_B, *[R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu])*Errors.Error_E(x, E_B, dE_B)

    part1_B = df.gl3_integrate(SignalRate.SR_B, ranges_B, percision, args=[R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu])

    part2_B = df.gl_integrate(fun1_B, E_min_B, E_max)

    part3_B = B_B/2

    ret_B = - part1_B + np.sum(np.log(part3_B + part2_B))

    return ret_B


# Total
def log_likelihood(theta, 
                   t_K, E_K, c_K, dE_K, B_K, 
                   t_I, E_I, c_I, dE_I, B_I,
                   t_B, E_B, c_B, dE_B, B_B
                   ):

    ret = (
        log_likelihood_K(theta, t_K, E_K, c_K, dE_K, B_K) +
           log_likelihood_I(theta, t_I, E_I, c_I, dE_I, B_I) +
           log_likelihood_B(theta, t_B, E_B, c_B, dE_B, B_B)
           )


    return ret


N_data = Data.Kam_E_valid.shape[0] + Data.IMB_E_valid.shape[0] + Data.Baksan_E_valid.shape[0]

print("Total number of data points: ", N_data)


constraints = []




Total number of data points:  24


In [17]:

def BIC_test(theta, k): # k the number of parameters
    return -2*log_likelihood(theta
                                         , Data.Kam_t_valid, Data.Kam_E_valid, Data.Kam_c_valid, Data.Kam_dE_valid, Data.Kam_B_valid_2009
                                         , Data.IMB_t_valid, Data.IMB_E_valid, Data.IMB_c_valid, Data.IMB_dE_valid, Data.IMB_B_valid
                                         , Data.Baksan_t_valid, Data.Baksan_E_valid, Data.Baksan_c_valid, Data.Baksan_dE_valid, Data.Baksan_B_valid
                                          ) + k*np.log(N_data)

def AIC_test(theta, k):
    return -2*log_likelihood(theta
                                         , Data.Kam_t, Data.Kam_E, Data.Kam_c, Data.Kam_dE, Data.Kam_B_2009
                                         , Data.IMB_t, Data.IMB_E, Data.IMB_c, Data.IMB_dE, Data.IMB_B
                                         , Data.Baksan_t, Data.Baksan_E, Data.Baksan_c, Data.Baksan_dE, Data.Baksan_B
                                          ) + k**2

def XI_square(theta):
    return -2*log_likelihood(theta
                                         , Data.Kam_t_valid, Data.Kam_E_valid, Data.Kam_c_valid, Data.Kam_dE_valid, Data.Kam_B_valid_2009
                                         , Data.IMB_t_valid, Data.IMB_E_valid, Data.IMB_c_valid, Data.IMB_dE_valid, Data.IMB_B_valid
                                         , Data.Baksan_t_valid, Data.Baksan_E_valid, Data.Baksan_c_valid, Data.Baksan_dE_valid, Data.Baksan_B_valid
                                          )



D1_1 = [10.63, 4.65, 4.63, 0.05, 1.91, 0.44, 1.63, -4.18]
D1_0 = [10.63, 4.65, 4.63, 0.05, 1.91, 0.44, 1.63, -17.18]
print(XI_square(D1_1), XI_square(D1_0))

3946.1303067952235 256.34907229262046


In [18]:
from Pycode import definition as df
import numpy as np
import matplotlib.pyplot as plt
from Pycode import SignalRate_2d as SignalRate
import Data
from scipy import integrate
from Pycode import Errors
from Simulation_Spectrum import time_limit


def log_likelihood_K(theta, t_K, E_K, c_K, dE_K, B_K):

    # R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu = theta

    R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu = theta

    fun1_K = lambda x: SignalRate.SR_K(t_K, x, c_K, *[R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu]) * Errors.Error_E(x, E_K, dE_K)        # optional

    part1_K = df.gl3_integrate(SignalRate.SR_K, ranges_K, percision, args=[R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu])

    part2_K = df.gl_integrate(fun1_K, E_min_K, E_max)

    part3_K = B_K/2

    ret_K = -part1_K + np.sum(np.log(part3_K +part2_K))

    return ret_K


# IMB
def log_likelihood_I(theta, t_I, E_I, c_I, dE_I, B_I):

    R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu = theta
    
    fun1_I = lambda x: SignalRate.SR_I(t_I, x, c_I, *[R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu])*Errors.Error_E(x, E_I, dE_I)

    fun2_I = lambda x, c, t: SignalRate.SR_I(t, x, c, *[R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu])*Errors.Error_E(x, E_I, dE_I)

    part1_I = df.gl3_integrate(SignalRate.SR_I, ranges_I, percision, args=[R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu])

    part2_I = np.clip(df.gl_integrate(fun1_I, E_min_I , E_max), 1e-100, None)

    part3_I = B_I/2

    part4_I= df.gl2_integrate_vec(fun2_I, ((E_min_I, E_max), (-1,1)), t_I, percision)

    ret_I = -0.9055*part1_I  + np.sum(0.035*part4_I) + np.sum(np.log(part3_I + part2_I))

    return ret_I

# Baksan
def log_likelihood_B(theta, t_B, E_B, c_B, dE_B, B_B):

    R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu= theta

    fun1_B = lambda x: SignalRate.SR_B(t_B, x, c_B, *[R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu])*Errors.Error_E(x, E_B, dE_B)

    part1_B = df.gl3_integrate(SignalRate.SR_B, ranges_B, percision, args=[R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu])

    part2_B = df.gl_integrate(fun1_B, E_min_B, E_max)

    part3_B = B_B/2

    ret_B = - part1_B + np.sum(np.log(part3_B + part2_B))

    return ret_B


# Total
def log_likelihood(theta, 
                   t_K, E_K, c_K, dE_K, B_K, 
                   t_I, E_I, c_I, dE_I, B_I,
                   t_B, E_B, c_B, dE_B, B_B
                   ):

    ret = (
        log_likelihood_K(theta, t_K, E_K, c_K, dE_K, B_K) +
           log_likelihood_I(theta, t_I, E_I, c_I, dE_I, B_I) +
           log_likelihood_B(theta, t_B, E_B, c_B, dE_B, B_B)
           )


    return ret


N_data = Data.Kam_E.shape[0] + Data.IMB_E.shape[0] + Data.Baksan_E.shape[0]

print("Total number of data points: ", N_data)

constraints = []

Total number of data points:  17


In [21]:
def BIC_test(theta, k): # k the number of parameters
    return -2*log_likelihood(theta
                                         , Data.Kam_t, Data.Kam_E, Data.Kam_c, Data.Kam_dE, Data.Kam_B_2009
                                         , Data.IMB_t, Data.IMB_E, Data.IMB_c, Data.IMB_dE, Data.IMB_B
                                         , Data.Baksan_t, Data.Baksan_E, Data.Baksan_c, Data.Baksan_dE, Data.Baksan_B
                                          ) + k*np.log(N_data)


def XI_square(theta):
    return -2*log_likelihood(theta
                                         , Data.Kam_t, Data.Kam_E, Data.Kam_c, Data.Kam_dE, Data.Kam_B_2009
                                         , Data.IMB_t, Data.IMB_E, Data.IMB_c, Data.IMB_dE, Data.IMB_B
                                         , Data.Baksan_t, Data.Baksan_E, Data.Baksan_c, Data.Baksan_dE, Data.Baksan_B
                                          )


D2_1 = [0, 0, 0, 0, 0, 0, 2.25, -6]
D2_0 = [0, 0, 0, 0, 0, 0, 2.25, -15.61]
print(XI_square(D2_1), XI_square(D2_0))

210.17731911484964 214.17956235303382
